## What is RAG?

RAG (retrieval-augmented generation) improves LLM answers by adding relevant external knowledge at query time, especially private or newer data not in model training.

### RAG architecture
A typical RAG app has two parts:

- **Indexing**: ingest, split, embed, and store documents (usually offline).

    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/WEE3pjeJvSZP0R7UL7CYTA.png" width="50%" alt="indexing"/>  <br>
    <span style="font-size: 10px;">[source](https://python.langchain.com/docs/tutorials/rag/)</span>

    1. Load: First, you must load your data. This is done with [DocumentLoaders](https://python.langchain.com/docs/how_to/#document-loaders).

    2. Split: [Text splitters](https://python.langchain.com/docs/how_to/#text-splitters) break large `Documents` into smaller chunks. This is useful both for indexing data and for passing it into a model because large chunks are harder to search and won’t fit in a model’s finite context window.

    3. Store: You need somewhere to store and index your splits so that they can later be searched. This is often done using a [VectorStore](https://python.langchain.com/docs/how_to/#vector-stores) and [Embeddings](https://python.langchain.com/docs/how_to/embed_text/) model.

- **Retrieval + generation**: at runtime, retrieve relevant chunks for a user query and pass them to the LLM to generate grounded answers.

    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/SwPO26VeaC8VTZwtmWh5TQ.png" width="50%" alt="retrieval"/> <br>
    <span style="font-size: 10px;">[source](https://python.langchain.com/docs/use_cases/question_answering/)</span>

    1. Retrieve: Given a user input, relevant splits are retrieved from storage using a retriever.
    2. Generate: A ChatModel / LLM produces an answer using a prompt that includes the question and the retrieved data.


In [5]:
# Install required libraries for Hugging Face + LangChain RAG (no Watsonx)
!pip3 -q install -U wget langchain langchain-core langchain-community langchain-text-splitters langchain-chroma langchain-huggingface chromadb sentence-transformers transformers huggingface-hub python-dotenv torch pypdf --break-system-packages

In [9]:
# Core Python
import os
import warnings
import wget

# Environment loading
from dotenv import load_dotenv

# LangChain document loading/splitting
from langchain_community.document_loaders import TextLoader, PyPDFLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

# Embeddings + vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# RAG chains + prompting + memory
from langchain_classic.chains import RetrievalQA
from langchain_classic.chains.conversational_retrieval.base import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate
from langchain_classic.memory import ConversationBufferMemory

# Hugging Face chat model wrappers
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

warnings.filterwarnings("ignore")
load_dotenv()

True

### Preprocessing

##### Load the Document

In [10]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)
print('file downloaded')

with open(filename, 'r') as file:
    # Read the contents of the file
    contents = file.read()
    print(contents)

file downloaded
1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potentia

##### Splitting the document into chunks

In [11]:
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts))

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16


##### Embedding & Storing

In [13]:
embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)  # store the embedding in docsearch using Chromadb
print('document ingested')

document ingested


### Integrating LangChain


In [14]:
minimax_llm = ChatHuggingFace(llm=HuggingFaceEndpoint(
    repo_id="MiniMaxAI/MiniMax-M2.5",
    task="conversational"
), temperature=0.8)

glm_llm = ChatHuggingFace(llm=HuggingFaceEndpoint(
    repo_id="zai-org/GLM-5",
    provider="novita",
    task="conversational"
), temperature=0.8)

In [15]:
qa = RetrievalQA.from_chain_type(llm=minimax_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "what is mobile policy?"
qa.invoke(query)

{'query': 'what is mobile policy?',
 'result': "Based on the provided document, the **Mobile Phone Policy** is a set of guidelines that establishes standards for the appropriate and responsible use of mobile devices within the organization. Its purpose is to ensure employees use mobile phones in a manner consistent with company values and legal compliance.\n\nThe policy covers several key areas:\n\n1. **Acceptable Use** – Mobile devices are primarily for work-related tasks, though limited personal usage is allowed if it doesn't disrupt work obligations.\n\n2. **Security** – Employees must safeguard their devices and login credentials, be cautious about downloading apps or clicking links from unknown sources, and promptly report any security concerns.\n\n3. **Confidentiality** – Avoid sending sensitive company information through unsecured channels and be discreet when discussing company matters in public.\n\n4. **Cost Management** – Keep personal phone usage separate from company accou

Asking a more higher-level question

In [16]:
qa = RetrievalQA.from_chain_type(llm=minimax_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "Can you summarize the document for me?"
qa.invoke(query)

{'query': 'Can you summarize the document for me?',
 'result': "Based on the document provided, here is a summary:\n\nThe document appears to be a set of organizational policies consisting of:\n\n1. **Code of Conduct** - This outlines the fundamental ethical standards for all members of the organization, including:\n   - **Integrity** - Acting honestly and transparently, protecting sensitive information, and avoiding conflicts of interest\n   - **Respect** - Embracing diversity, creating an inclusive environment, and prohibiting discrimination or harassment\n   - **Accountability** - Taking responsibility for actions, following laws and regulations, and reporting violations\n   - **Safety** - Prioritizing the safety of employees, clients, and the community\n   - **Environmental Responsibility** - Minimizing environmental impact and promoting sustainable practices\n\n2. **Health and Safety Policy** - A commitment to maintaining a workplace free from hazards, complying with relevant laws

With another model

In [17]:
qa = RetrievalQA.from_chain_type(llm=glm_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "Can you summarize the document for me?"
qa.invoke(query)

{'query': 'Can you summarize the document for me?', 'result': ''}

##### Using Prompt Template

Dive deeper to let the model realise and reply if it doesn't exists.

For example, a information that does not exist in document,

In [25]:
qa = RetrievalQA.from_chain_type(llm=minimax_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': 'Based on the provided policy documents, there is no specific rule mentioned regarding eating in company vehicles.\n\nThe only reference to company vehicles in the Smoking Policy states that "smoking is not permitted in company vehicles, whether they are owned or leased, to maintain the condition and cleanliness of these vehicles." This suggests the company values maintaining cleanliness in company vehicles, but there is no explicit mention of whether eating is allowed or prohibited.\n\nI don\'t know the specific answer to your question. You may want to check with your supervisor or HR department for clarification on whether eating is permitted in company vehicles.'}

Now letting it know via Prompt Template to answer if it doesn't know (for future questions),

In [21]:
prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definately do not try to make up an answer.

{context}

Question: {question}
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

In [26]:
qa = RetrievalQA.from_chain_type(llm=minimax_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 chain_type_kwargs=chain_type_kwargs, 
                                 return_source_documents=False)

query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': "Based on the document provided, I don't know. The policies address that smoking is not permitted in company vehicles, but there is no information provided about whether eating is allowed in company vehicles."}

##### Make the conversation have memory
Take a look at a situation in which an LLM does not have memory.

In [28]:
query = "What I cannot do in it?"
qa.invoke(query)

{'question': 'What I cannot do in it?',
 'chat_history': '',
 'answer': "Based on the policies provided, here are the things you **cannot do**:\n\n## Internet and Email Policy Restrictions\n\n- **Share your login credentials or passwords** with others\n- **Open attachments or click links from unknown/untrusted sources**\n- **Transmit confidential information** (trade secrets, sensitive customer data) **without encryption**\n- **Discuss company matters on public forums or social media** without discretion\n- **Engage in harassment, discrimination**, or distribute offensive/inappropriate content\n- **Use internet/email for personal purposes** that interfere with your work responsibilities\n- **Violate copyright or data protection laws**\n\n## Mobile Phone Policy Restrictions\n\n- **Use mobile devices in a way that disrupts your work obligations**\n- **Download apps or click links from unfamiliar sources**\n- **Transmit sensitive company information via unsecured messaging apps or emails*

Adding memory buffer

In [29]:
memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)
qa = ConversationalRetrievalChain.from_llm(llm=minimax_llm, 
                                           chain_type="stuff", 
                                           retriever=docsearch.as_retriever(), 
                                           memory = memory, 
                                           get_chat_history=lambda h : h, 
                                           return_source_documents=False)

In [30]:
history = []
query = "What is mobile policy?"
result = qa.invoke({"question":query}, {"chat_history": history})
print(result["answer"])
history.append((query, result["answer"]))

Based on the provided document, the **Mobile Phone Policy** sets forth the standards and expectations for the appropriate and responsible usage of mobile devices within the organization.

Its main purpose is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance.

The key guidelines include:

- **Acceptable Use:** Mobile devices are primarily for work-related tasks, with limited personal usage allowed as long as it doesn't disrupt work obligations.
- **Security:** Employees must safeguard their devices and login credentials, be cautious when downloading apps or clicking links from unfamiliar sources, and promptly report any security concerns.
- **Confidentiality:** Avoid sending sensitive company information through unsecured channels and be discreet when discussing company matters in public.
- **Cost Management:** Keep personal phone usage separate from company accounts and reimburse the company for any personal charges on compan

In [31]:
query = "List points in it?"
result = qa({"question": query}, {"chat_history": history})
print(result["answer"])

Based on the Mobile Phone Policy, here are the key points:

1. **Acceptable Use**: Mobile devices are primarily for work-related tasks. Limited personal use is allowed as long as it doesn't interfere with work obligations.

2. **Security**: Protect your device and access credentials. Be cautious when downloading apps or clicking links from unknown sources. Report any security concerns or suspicious activities immediately.

3. **Confidentiality**: Avoid sending sensitive company information through unsecured messaging apps or emails. Be discreet when discussing company matters in public places.

4. **Cost Management**: Keep personal phone usage separate from company accounts. Reimburse the company for any personal charges on company-issued phones.

5. **Compliance**: Follow all relevant laws and regulations regarding mobile phone usage, including data protection and privacy laws.

6. **Lost or Stolen Devices**: Report lost or stolen devices to the IT department or your supervisor immedi

##### Wrapping it as an Agent

Serves as a session having memory of previous conversation and context

In [33]:
def qa():
    memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)
    qa = ConversationalRetrievalChain.from_llm(llm=minimax_llm, 
                                               chain_type="stuff", 
                                               retriever=docsearch.as_retriever(), 
                                               memory = memory, 
                                               get_chat_history=lambda h : h, 
                                               return_source_documents=False)
    history = []
    while True:
        query = input("Question: ")
        
        if query.lower() in ["quit","exit","bye"]:
            print("Answer: Goodbye!")
            break
            
        result = qa({"question": query}, {"chat_history": history})
        
        history.append((query, result["answer"]))
        
        print("Answer: ", result["answer"])

In [34]:
qa()

Answer:  I can see you've shared a list of policy titles:

1. Smoking Policy
2. Drug and Alcohol Policy
3. Recruitment Policy
4. Mobile Phone Policy

However, you haven't asked a specific question. How would you like me to help you with these policies? For example, I can:

- Help you draft or expand on one of these policies
- Review existing policy text you have
- Create a table of contents for a policy handbook
- Answer questions about what these policies typically include

Please let me know what you'd like to do!
Answer: Goodbye!


### Exercise 1: Work on your own document

You are welcome to use your own document to practice. Another document has also been prepared that you can use for practice. Can you load this document and make the LLM read it for you? <br>
Here is the URL to the document: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XVnuuEg94sAE4S_xAsGxBA.txt


In [36]:
loader = WebBaseLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XVnuuEg94sAE4S_xAsGxBA.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts))

embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)  # store the embedding in docsearch using Chromadb
print('document ingested')

qa = RetrievalQA.from_chain_type(llm=minimax_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "What is this document about?"
qa.invoke(query)

42
document ingested


{'query': 'What is this document about?',
 'result': "Based on the content provided, this appears to be a collection of HR policies from an organization. The document includes three main sections:\n\n1. **Recruitment Policy** - outlines how the organization attracts, selects, and hires new employees, emphasizing equal opportunity, transparency, and a fair selection process.\n\n2. **Code of Conduct** - establishes ethical guidelines for all employees, covering principles like integrity, respect, accountability, safety, and environmental responsibility.\n\n3. **Anti-Discrimination and Harassment Policy** - details the organization's commitment to a discrimination-free workplace, defines prohibited behaviors, and outlines reporting procedures and consequences.\n\nTogether, these policies form the foundation of the organization's approach to human resources, employee relations, and workplace culture. They aim to ensure a diverse, inclusive, ethical, and safe working environment."}

### Exercise 2: Return the source from the document
Sometimes, you not only want the LLM to summarize for you, but you also want the model to return the exact content source from the document to you for reference. Can you adjust the code to make it happen?

In [37]:
query = "What exact code of conducts?"
qa.invoke(query)

{'query': 'What exact code of conducts?',
 'result': "Based on the provided context, the Code of Conduct includes the following key principles:\n\n1. **Integrity**: Acting honestly and transparently in all interactions, respecting and protecting sensitive information, and avoiding conflicts of interest.\n\n2. **Respect**: Embracing diversity, valuing each individual's contributions, and maintaining an inclusive environment where everyone is treated with dignity and courtesy. Discrimination, harassment, or disrespectful behavior is unacceptable.\n\n3. **Accountability**: Taking responsibility for actions and decisions, following relevant laws and regulations, and continuously improving practices. This includes reporting potential violations and supporting investigations.\n\n4. **Safety**: Prioritizing the safety of employees, clients, and the communities served by maintaining a culture of safety and reporting unsafe conditions or practices.\n\n5. **Environmental Responsibility**: Commit

### Exercise 3: Use another LLM model

IBM watsonx.ai also has many other LLM models that you can use; for example, `mistralai/mistral-small-3-1-24b-instruct-2503`, an open-source model from Mistral AI. Can you change the model to see the difference of the response?

In [38]:
qa = RetrievalQA.from_chain_type(llm=glm_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "What exact code of conducts?"
qa.invoke(query)

{'query': 'What exact code of conducts?',
 'result': 'Based on the provided context, the Code of Conduct outlines the following five fundamental principles:\n\n1.  **Integrity:** Acting honestly and transparently, protecting sensitive information, and avoiding conflicts of interest.\n2.  **Respect:** Embracing diversity, prohibiting discrimination and harassment, and treating everyone with dignity.\n3.  **Accountability:** Taking responsibility for'}